# Sales Forecasting Pipeline

End-to-end ML pipeline that ingests historical sales data, engineers features,
trains a gradient-boosting forecasting model, and produces 90-day forward projections
with confidence intervals. Used by the supply-chain team for inventory planning.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.pipeline import Pipeline
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
FORECAST_HORIZON = 90   # days
N_SPLITS = 5            # time-series CV folds
TARGET_COL = 'units_sold'
DATE_COL = 'date'

print('Libraries loaded.')

Libraries loaded.


## 1. Data Ingestion
Load and validate the raw sales CSV.

In [2]:
np.random.seed(RANDOM_STATE)
dates = pd.date_range('2020-01-01', '2023-12-31', freq='D')
trend = np.linspace(500, 900, len(dates))
seasonality = 120 * np.sin(2 * np.pi * np.arange(len(dates)) / 365)
weekly = 40 * np.sin(2 * np.pi * np.arange(len(dates)) / 7)
noise = np.random.normal(0, 30, len(dates))
covid_dip = np.where((dates >= '2020-03-01') & (dates <= '2020-06-30'), -150, 0)

df = pd.DataFrame({
    DATE_COL: dates,
    TARGET_COL: np.maximum(0, trend + seasonality + weekly + noise + covid_dip).astype(int),
    'price': np.random.uniform(18, 42, len(dates)).round(2),
    'promotion': np.random.choice([0, 1], len(dates), p=[0.85, 0.15]),
    'category': np.random.choice(['A', 'B', 'C'], len(dates), p=[0.5, 0.3, 0.2]),
    'region': np.random.choice(['North', 'South', 'East', 'West'], len(dates)),
})

print(f'Dataset: {len(df):,} rows, {df.columns.tolist()}')
print(df.describe())

## 2. Feature Engineering
Create lag features, rolling statistics, and calendar features.

In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values(DATE_COL)
    df['day_of_week'] = df[DATE_COL].dt.dayofweek
    df['day_of_month'] = df[DATE_COL].dt.day
    df['month'] = df[DATE_COL].dt.month
    df['quarter'] = df[DATE_COL].dt.quarter
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['week_of_year'] = df[DATE_COL].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28, 90]:
        df[f'lag_{lag}'] = df[TARGET_COL].shift(lag)
    for window in [7, 14, 28]:
        df[f'rolling_mean_{window}'] = df[TARGET_COL].shift(1).rolling(window).mean()
        df[f'rolling_std_{window}']  = df[TARGET_COL].shift(1).rolling(window).std()
    for col in ['category', 'region']:
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat([df, dummies], axis=1)
    df = df.dropna()
    return df

df_feat = engineer_features(df)
feature_cols = [c for c in df_feat.columns
                if c not in [DATE_COL, TARGET_COL, 'category', 'region']]
print(f'Features: {len(feature_cols)}')

## 3. Model Training with Time-Series Cross-Validation

In [4]:
X = df_feat[feature_cols]
y = df_feat[TARGET_COL]

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('gbr', GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        min_samples_leaf=10,
        random_state=RANDOM_STATE
    ))
])

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
cv_scores = cross_val_score(pipeline, X, y, cv=tscv,
                             scoring='neg_mean_absolute_percentage_error')
print(f'CV MAPE: {-cv_scores.mean():.2%} ± {cv_scores.std():.2%}')

pipeline.fit(X, y)
train_preds = pipeline.predict(X)
train_mape = mean_absolute_percentage_error(y, train_preds)
print(f'Train MAPE: {train_mape:.2%}')

## 4. 90-Day Forecast Generation

In [5]:
last_date = df_feat[DATE_COL].max()
forecast_dates = pd.date_range(last_date + pd.Timedelta(days=1),
                               periods=FORECAST_HORIZON, freq='D')

future_df = pd.DataFrame({DATE_COL: forecast_dates})
future_df['price']     = df['price'].tail(30).mean()
future_df['promotion'] = 0
future_df['category']  = 'A'
future_df['region']    = 'North'

all_df = pd.concat([df, future_df], ignore_index=True)
all_df_feat = engineer_features(all_df)
future_feat = all_df_feat[all_df_feat[DATE_COL].isin(forecast_dates)]

predictions   = pipeline.predict(future_feat[feature_cols])
lower_bound   = predictions * 0.85
upper_bound   = predictions * 1.15

forecast_df = pd.DataFrame({
    'date': forecast_dates,
    'forecast': predictions.round(),
    'lower_80ci': lower_bound.round(),
    'upper_80ci': upper_bound.round(),
})

print(forecast_df.head(10).to_string(index=False))

## 5. Feature Importance & Export

In [6]:
importances = pd.Series(
    pipeline.named_steps['gbr'].feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print('Top 10 features:')
print(importances.head(10))

output_path = Path('outputs')
output_path.mkdir(exist_ok=True)
forecast_df.to_csv(output_path / 'sales_forecast_90d.csv', index=False)
print(f'\nForecast saved → {output_path}/sales_forecast_90d.csv')